In [ ]:
dec_real = result_real.get_decoder_importance()
dec_swap = result_swap.get_decoder_importance()

print(type(dec_real))

if isinstance(dec_real, list):
    print(f"List of {len(dec_real)} series")
    print(f"First item type : {type(dec_real[0])}")
    print(f"First item shape: {dec_real[0].shape}")
    print(dec_real[0].head())
elif hasattr(dec_real, 'shape'):
    print(f"DataFrame shape: {dec_real.shape}")
    print(dec_real.head())

In [ ]:
step_labels = {0: 'June_2026', 1: 'July_2026', 2: 'August_2026'}

per_step_real = {}
per_step_swap = {}

for step_idx, step_name in step_labels.items():
    per_step_real[step_name] = pd.concat(
        [dec_real[i].iloc[[step_idx]] for i in range(len(dec_real))]
    ).mean(axis=0)

    per_step_swap[step_name] = pd.concat(
        [dec_swap[i].iloc[[step_idx]] for i in range(len(dec_swap))]
    ).mean(axis=0)

In [ ]:
per_step_real = {
    'June_2026'  : dec_real.iloc[0],
    'July_2026'  : dec_real.iloc[1],
    'August_2026': dec_real.iloc[2]
}
per_step_swap = {
    'June_2026'  : dec_swap.iloc[0],
    'July_2026'  : dec_swap.iloc[1],
    'August_2026': dec_swap.iloc[2]
}

In [ ]:
for step_name in ['June_2026', 'July_2026', 'August_2026']:
    comp = pd.DataFrame({
        'original': per_step_real[step_name],
        'swapped' : per_step_swap[step_name]
    })
    comp['shift'] = comp['swapped'] - comp['original']
    comp = comp.sort_values('original', ascending=False)

    print(f"\n{'='*60}")
    print(f"  {step_name}")
    print(f"{'='*60}")
    print(comp.head(10).round(4))
    print(f"\nTop gainers when swapped (positive shift):")
    print(comp.sort_values('shift', ascending=False).head(5).round(4))

In [ ]:
print([m for m in dir(result_real) if not m.startswith('_')])

In [ ]:
def run_single_month_swap(target_month, source_month, col_list=future_covariates):
    swapped_df = lookahead_data_pandas_df.copy()
    swapped_df.index = pd.to_datetime(swapped_df.index)

    src = swapped_df[swapped_df.index == pd.Timestamp(source_month)]\
              .set_index('PARENT_DEALER_CODE_MODEL_FAMILY')[col_list]
    mask = swapped_df.index == pd.Timestamp(target_month)
    grp  = swapped_df.loc[mask, 'PARENT_DEALER_CODE_MODEL_FAMILY'].values
    nv   = src.reindex(grp)[col_list].values.astype(float)
    ov   = swapped_df.loc[mask, col_list].values.astype(float)
    nan  = np.isnan(nv); nv[nan] = ov[nan]
    swapped_df.loc[mask, col_list] = nv

    ts = TimeSeries.from_group_dataframe(
        df=swapped_df, group_cols='PARENT_DEALER_CODE_MODEL_FAMILY',
        static_cols=static_covariates, value_cols=future_covariates, freq='MS')
    scaled = static_transformer.transform(
                 future_covariates_scaler.transform(ts))
    return scaled

# Three separate single-month swaps
la_june_only = run_single_month_swap('2026-06-01', '2025-06-01')
la_july_only = run_single_month_swap('2026-07-01', '2025-07-01')
la_aug_only  = run_single_month_swap('2026-08-01', '2025-08-01')

N = 100
results = {}
for label, la in [('June_swap',  la_june_only),
                   ('July_swap',  la_july_only),
                   ('August_swap', la_aug_only)]:
    exp = TFTExplainer(loaded_model,
                       background_series=final_scaled_lookback_data[:N],
                       background_future_covariates=la[:N])
    res = exp.explain(foreground_series=final_scaled_lookback_data[:N],
                      foreground_future_covariates=la[:N])
    dec = res.get_decoder_importance()
    if isinstance(dec, list):
        dec = pd.concat(dec).groupby(level=0).mean()
    results[label] = dec.mean(axis=0)

# Original for comparison
results['Original'] = pd.concat(
    result_real.get_decoder_importance()
).groupby(level=0).mean()

comparison = pd.DataFrame(results).sort_values('Original', ascending=False)
comparison['June_shift']   = comparison['June_swap']   - comparison['Original']
comparison['July_shift']   = comparison['July_swap']   - comparison['Original']
comparison['August_shift'] = comparison['August_swap'] - comparison['Original']

print(comparison[['Original','June_shift','July_shift','August_shift']].head(15).round(2))

In [ ]:
['available_components', 'explained_components', 'feature_importances', 'get_attention', 'get_decoder_importance', 'get_encoder_importance', 'get_explanation', 'get_feature_importances', 'get_static_covariates_importance']

In [ ]:
# Use whatever name appears for decoder in the output above
dec_explanation = result_real.get_explanation('decoder_importance')  # adjust name
print(type(dec_explanation))
print(dec_explanation.shape if hasattr(dec_explanation, 'shape') else type(dec_explanation))

In [ ]:
print(len(dec_explanation))
print(type(dec_explanation[0]))

# If it's a Darts TimeSeries
if hasattr(dec_explanation[0], 'pd_dataframe'):
    df = dec_explanation[0].pd_dataframe()
    print(df.shape)
    print(df)

# If it's already a DataFrame
elif hasattr(dec_explanation[0], 'shape'):
    print(dec_explanation[0].shape)
    print(dec_explanation[0])

In [ ]:
def run_single_month_swap(target_month, source_month, col_list=future_covariates):
    swapped_df = lookahead_data_pandas_df.copy()
    swapped_df.index = pd.to_datetime(swapped_df.index)

    src = swapped_df[swapped_df.index == pd.Timestamp(source_month)]\
              .set_index('PARENT_DEALER_CODE_MODEL_FAMILY')[col_list]
    mask = swapped_df.index == pd.Timestamp(target_month)
    grp  = swapped_df.loc[mask, 'PARENT_DEALER_CODE_MODEL_FAMILY'].values
    nv   = src.reindex(grp)[col_list].values.astype(float)
    ov   = swapped_df.loc[mask, col_list].values.astype(float)
    nan  = np.isnan(nv); nv[nan] = ov[nan]
    swapped_df.loc[mask, col_list] = nv

    ts = TimeSeries.from_group_dataframe(
        df=swapped_df, group_cols='PARENT_DEALER_CODE_MODEL_FAMILY',
        static_cols=static_covariates, value_cols=future_covariates, freq='MS')
    scaled = static_transformer.transform(
                 future_covariates_scaler.transform(ts))
    return scaled

# Three separate single-month swaps
la_june_only = run_single_month_swap('2026-06-01', '2025-06-01')
la_july_only = run_single_month_swap('2026-07-01', '2025-07-01')
la_aug_only  = run_single_month_swap('2026-08-01', '2025-08-01')

N = 100
results = {}
for label, la in [('June_swap',  la_june_only),
                   ('July_swap',  la_july_only),
                   ('August_swap', la_aug_only)]:
    exp = TFTExplainer(loaded_model,
                       background_series=final_scaled_lookback_data[:N],
                       background_future_covariates=la[:N])
    res = exp.explain(foreground_series=final_scaled_lookback_data[:N],
                      foreground_future_covariates=la[:N])
    dec = res.get_decoder_importance()
    if isinstance(dec, list):
        dec = pd.concat(dec).groupby(level=0).mean()
    results[label] = dec.mean(axis=0)

# Original for comparison
results['Original'] = pd.concat(
    result_real.get_decoder_importance()
).groupby(level=0).mean()

comparison = pd.DataFrame(results).sort_values('Original', ascending=False)
comparison['June_shift']   = comparison['June_swap']   - comparison['Original']
comparison['July_shift']   = comparison['July_swap']   - comparison['Original']
comparison['August_shift'] = comparison['August_swap'] - comparison['Original']

print(comparison[['Original','June_shift','July_shift','August_shift']].head(15).round(2))